# Phân Cụm Khách Hàng Bán Lẻ Trực Tuyến bằng Thuật Toán Phân Cấp Tích Lũy (Agglomerative Hierarchical Clustering)

## 📌 Giới Thiệu Đồ Án & Bài Toán Phân Khúc Khách Hàng
Trong ngành bán lẻ trực tuyến (**Online Retail**), việc hiểu rõ hành vi khách hàng là chìa khóa để tối ưu hóa doanh số và giữ chân người mua. Chúng ta mô hình hóa hành vi khách hàng thông qua bộ 3 chỉ số kinh điển **RFM**:
- **Recency ($R$) - Tính gần đây**: Số ngày trôi qua kể từ lần mua sắm cuối cùng của khách hàng (càng nhỏ càng tốt, chứng tỏ khách mới mua gần đây).
- **Frequency ($F$) - Tần suất**: Tổng số lượng hóa đơn/đơn hàng hợp lệ đã mua trong quá khứ (càng lớn càng tốt).
- **Monetary ($M$) - Giá trị chi tiêu**: Tổng số tiền khách hàng đã chi tiêu (£) (càng lớn càng tốt).

### 🎯 Mục Tiêu Nghiên Cứu Của Notebook:
1. **Khám phá cấu trúc cụm từ dưới lên (Bottom-up)** bằng thuật toán **Agglomerative Clustering**.
2. **Thử nghiệm 4 phương pháp liên kết (Linkages)**: `Ward` (tối thiểu hóa phương sai), `Complete` (khoảng cách cực đại), `Average` (khoảng cách trung bình), và `Single` (khoảng cách cực tiểu / hiệu ứng xâu chuỗi).
3. **Thử nghiệm dải số cụm $k \in [2, 10]$** và đánh giá khách quan qua 3 chỉ số đo lường nội suy: **Silhouette Score**, **Davies-Bouldin Index**, và **Calinski-Harabasz Index**.
4. **Vẽ cây phả hệ Dendrogram**, xác định điểm cắt ngang (Horizontal Cut Line) và phân tích khoảng nhảy khoảng cách (Distance Gap).
5. **Phác họa chân dung khách hàng (Customer Persona Profiling)**: Thống kê chi tiết RFM gốc và trực quan hóa qua PCA 2D và Boxplots.
6. **Kiểm định ngoài thời gian (Out-of-time Temporal Evaluation)**: Theo dõi hành vi mua lại thực tế của từng cụm trong **90 ngày tiếp theo**.

In [ ]:
# 1. Cài đặt và nạp các thư viện cần thiết
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

# Hỗ trợ hiển thị bảng dữ liệu (tương thích cả Jupyter Notebook và script thông thường)
try:
    from IPython.display import display
except Exception:
    def display(*args):
        for arg in args:
            if hasattr(arg, 'to_string'):
                print(arg.to_string())
            else:
                print(arg)

# Tự động xác định đường dẫn gốc của repository chứa thư mục 'src'
ROOT = Path.cwd().resolve()
if not (ROOT / 'data_cleaned').exists() and (ROOT / 'Online-Retail-Customer-Segmentation' / 'data_cleaned').exists():
    ROOT = (ROOT / 'Online-Retail-Customer-Segmentation').resolve()
elif not (ROOT / 'data_cleaned').exists() and Path('D:/WORKSPACE/Online-Retail-Customer-Segmentation/data_cleaned').exists():
    ROOT = Path('D:/WORKSPACE/Online-Retail-Customer-Segmentation').resolve()

for p_add in (str(ROOT), str(ROOT / 'src')):
    if p_add not in sys.path:
        sys.path.insert(0, p_add)

try:
    from retail_segmentation.pipeline import FEATURE_COLUMNS, RFM_COLUMNS
    from retail_segmentation.temporal_evaluation import load_phase
except ImportError:
    fallback_src = Path('D:/WORKSPACE/Online-Retail-Customer-Segmentation/src').resolve()
    if str(fallback_src) not in sys.path:
        sys.path.insert(0, str(fallback_src))
    from retail_segmentation.pipeline import FEATURE_COLUMNS, RFM_COLUMNS
    from retail_segmentation.temporal_evaluation import load_phase

# Cấu hình giao diện đồ họa đẹp mắt
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams.update({'figure.dpi': 150, 'figure.autolayout': True})
print('✓ Đã nạp thành công các thư viện phân tích và module retail_segmentation!')

## 1. Nạp và Khám Phá Dữ Liệu RFM Chuẩn Hóa

Chúng ta sử dụng tập dữ liệu giai đoạn **Development Phase** (trước ngày 01/09/2011, gồm **2.716 khách hàng**).
Dữ liệu đầu vào của mô hình là ma trận $X$ gồm 3 đặc trưng đã qua phép biến đổi logarit tự nhiên `log1p` và chuẩn hóa z-score `StandardScaler`:
- `Recency_scaled`
- `Frequency_scaled`
- `Monetary_scaled`

> **Lưu ý quan trọng**: Phép biến đổi Logarithm và Chuẩn hóa Standard Scaling là điều kiện tiên quyết bắt buộc trước khi tính khoảng cách Euclidean, vì phân phối RFM trong bán lẻ luôn bị lệch phải nặng do các khách hàng chi tiêu lớn (VIPs/Whales).

In [ ]:
# Tải dữ liệu giai đoạn development
rfm_dev, feat_dev, outcomes_dev = load_phase(ROOT / 'data_cleaned', 'development')
X_dev = feat_dev[FEATURE_COLUMNS].to_numpy()

print(f'Tổng số khách hàng phân tích (Giai đoạn Development): {len(rfm_dev):,} khách hàng')
print('Kích thước ma trận đặc trưng huấn luyện X:', X_dev.shape)

print('\n--- 5 DÒNG ĐẦU TIÊN CỦA MA TRẬN ĐẶC TRƯNG CHUẨN HÓA ---')
display(feat_dev.head())

print('\n--- BẢNG THỐNG KÊ MÔ TẢ RFM GỐC CỦA KHÁCH HÀNG ---')
display(rfm_dev.describe())

## 2. Phân Tích Cây Phân Cấp (Dendrogram) & So Sánh Các Phương Pháp Liên Kết (Linkages)

### Nguyên Lý Của Thuật Toán Agglomerative (Tiếp Cận Từ Dưới Lên):
- **Khởi tạo**: Ban đầu, mỗi khách hàng được xem là 1 cụm riêng biệt ($N = 2.716$ cụm).
- **Lặp lại**: Ở mỗi bước, tính khoảng cách giữa tất cả các cặp cụm và gộp 2 cụm có khoảng cách nhỏ nhất thành một cụm mới. Số cụm giảm dần $N - 1, N - 2, \dots$
- **Kết thúc**: Sau $N-1$ bước gộp, toàn bộ khách hàng hội tụ về 1 cụm duy nhất.

### 4 Phương Pháp Liên Kết (Linkage Methods):
1. **Ward Linkage (`ward`)**: Tối thiểu hóa mức tăng tổng phương sai bên trong các cụm được gộp (Variance / Error Sum of Squares $\Delta ESS$). Ưu tiên tạo các cụm hình cầu lồi (spherical) có kích thước cân đối.
2. **Complete Linkage (`complete`)**: Khoảng cách giữa 2 cụm là khoảng cách xa nhất giữa 2 phần tử bất kỳ ($D(A, B) = \max \{d(x, y): x \in A, y \in B\}$). Tránh chuỗi dài, tạo cụm chặt chẽ.
3. **Average Linkage (`average`)**: Khoảng cách trung bình giữa tất cả các cặp phần tử của 2 cụm. Cân bằng giữa Ward và Complete.
4. **Single Linkage (`single`)**: Khoảng cách gần nhất giữa 2 cụm. Bị hiện tượng xâu chuỗi (Chaining Effect) - tạo thành dải dài, không thể phân nhóm.

In [ ]:
# Tính toán ma trận liên kết cho cả 4 phương pháp
print('Đang tính toán ma trận liên kết Linkage Matrices (scipy)...')
Z_ward = linkage(X_dev, method='ward', metric='euclidean')
Z_complete = linkage(X_dev, method='complete', metric='euclidean')
Z_average = linkage(X_dev, method='average', metric='euclidean')
Z_single = linkage(X_dev, method='single', metric='euclidean')

# Vẽ so sánh 4 cây phả hệ Dendrogram (rút gọn hiển thị 30 nút gộp cấp cao nhất)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

phuong_phap_list = [
    ('Ward (Tối thiểu hóa mức tăng phương sai)', Z_ward),
    ('Complete (Khoảng cách xa nhất)', Z_complete),
    ('Average (Khoảng cách trung bình)', Z_average),
    ('Single (Khoảng cách gần nhất - Hiệu ứng xâu chuỗi)', Z_single),
]

for idx, (ten_pp, Z) in enumerate(phuong_phap_list):
    dendrogram(
        Z,
        truncate_mode='lastp',
        p=30,
        leaf_rotation=90,
        leaf_font_size=9,
        show_contracted=True,
        ax=axes[idx],
    )
    axes[idx].set_title(ten_pp, fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Khoảng cách gộp (Merger Distance)')
    axes[idx].set_xlabel('Các nhánh cụm rút gọn (Số trong ngoặc là số khách hàng)')

plt.suptitle('So Sánh Cây Phả Hệ Dendrogram Giữa 4 Phương Pháp Liên Kết (Giai Đoạn Development)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Phân Tích Cây Ward Linkage & Xác Định Điểm Cắt Ngang (Horizontal Cut Line)

Trong cây phả hệ của **Ward Linkage**, trục tung biểu diễn mức gia tăng phương sai (Ward Distance $\Delta ESS$).
- **Nguyên tắc chọn đường cắt**: Tìm khoảng trống thẳng đứng dài nhất (**Largest Vertical Distance Gap**) không bị bất kỳ nhánh ngang nào cắt qua.
- Khoảng nhảy lớn nhất nằm trong khoảng cách từ $\sim 31.0$ đến $\sim 45.0$:
  - Đường cắt ngang tại ngưỡng $\sim 38.0$ cho ra đúng **$k = 4$ cụm** với kích thước rất cân đối ($215$, $319$, $980$, và $1.202$ khách hàng).
  - Đường cắt ngang tại ngưỡng $\sim 31.0$ cho ra **$k = 5$ cụm**.

In [ ]:
# Vẽ Dendrogram Ward với các đường cắt ngang lựa chọn số cụm
plt.figure(figsize=(13, 7))
ddata = dendrogram(
    Z_ward,
    truncate_mode='lastp',
    p=35,
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True,
    color_threshold=38.0,
)

plt.axhline(y=38.0, color='crimson', linestyle='--', linewidth=2, label='Đường cắt k=4 (Ngưỡng khoảng cách = 38.0)')
plt.axhline(y=31.0, color='darkorange', linestyle=':', linewidth=2, label='Đường cắt k=5 (Ngưỡng khoảng cách = 31.0)')

plt.title('Cây Phả Hệ Ward Linkage Với Các Đường Cắt Ngang Lựa Chọn Số Cụm k', fontsize=13, fontweight='bold')
plt.xlabel('Các nhánh phân cụm (Số trong ngoặc là quy mô số lượng khách hàng)')
plt.ylabel('Mức gia tăng phương sai Ward (ΔESS)')
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 3. Thử Nghiệm Tìm Kiếm Lưới (Grid Search) với $k \in [2, 10]$

Chúng ta tiến hành thử nghiệm toàn diện qua 3 phương pháp liên kết (`ward`, `complete`, `average`) và số lượng cụm $k$ từ 2 đến 10 dựa trên 3 thước đo nội suy chuẩn:
1. **Silhouette Score**: Đo lường sự tương đồng của một điểm với cụm của nó so với cụm gần nhất khác ([-1, 1], càng cao càng tốt).
2. **Davies-Bouldin Index**: Tỷ số giữa độ phân tán nội cụm và khoảng cách liên cụm (càng thấp càng tốt).
3. **Calinski-Harabasz Index**: Tỷ số giữa phương sai ngoại cụm và phương sai nội cụm (càng cao càng tốt).

In [ ]:
# Tìm kiếm lưới (Grid Search) qua các liên kết và số cụm k
linkages = ['ward', 'complete', 'average']
k_range = range(2, 11)
records = []

for method in linkages:
    for k in k_range:
        t0 = time.perf_counter()
        model = AgglomerativeClustering(n_clusters=k, linkage=method, metric='euclidean')
        labels = model.fit_predict(X_dev)
        runtime = time.perf_counter() - t0

        _, counts = np.unique(labels, return_counts=True)
        records.append({
            'Phương pháp liên kết': {'ward': 'Ward (Phương sai)', 'complete': 'Complete (Cực đại)', 'average': 'Average (Trung bình)'}[method],
            'Số cụm k': k,
            'Silhouette': round(float(silhouette_score(X_dev, labels)), 4),
            'Davies-Bouldin': round(float(davies_bouldin_score(X_dev, labels)), 4),
            'Calinski-Harabasz': round(float(calinski_harabasz_score(X_dev, labels)), 2),
            'Cụm nhỏ nhất': int(np.min(counts)),
            'Cụm lớn nhất': int(np.max(counts)),
            'Thời gian (giây)': round(runtime, 4),
        })

eval_df = pd.DataFrame(records)
print('--- BẢNG KẾT QUẢ ĐÁNH GIÁ TÌM KIẾM LƯỚI ---')
display(eval_df.head(15))

In [ ]:
# Vẽ biểu đồ so sánh 3 chỉ số đo lường giữa các phương pháp liên kết theo k
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {'Ward (Phương sai)': '#1f77b4', 'Complete (Cực đại)': '#2ca02c', 'Average (Trung bình)': '#ff7f0e'}
markers = {'Ward (Phương sai)': 'o', 'Complete (Cực đại)': 's', 'Average (Trung bình)': '^'}

# 1. Silhouette Score
for ten_m, sub in eval_df.groupby('Phương pháp liên kết'):
    axes[0].plot(sub['Số cụm k'], sub['Silhouette'], marker=markers[ten_m], label=ten_m, color=palette[ten_m])
axes[0].set_title('Hệ số Silhouette (Càng cao càng tốt)', fontweight='bold')
axes[0].set_xlabel('Số lượng cụm (k)')
axes[0].set_ylabel('Hệ số Silhouette')
axes[0].legend()

# 2. Davies-Bouldin Index
for ten_m, sub in eval_df.groupby('Phương pháp liên kết'):
    axes[1].plot(sub['Số cụm k'], sub['Davies-Bouldin'], marker=markers[ten_m], label=ten_m, color=palette[ten_m])
axes[1].set_title('Chỉ số Davies-Bouldin (Càng thấp càng tốt)', fontweight='bold')
axes[1].set_xlabel('Số lượng cụm (k)')
axes[1].set_ylabel('Chỉ số Davies-Bouldin')
axes[1].legend()

# 3. Calinski-Harabasz Index
for ten_m, sub in eval_df.groupby('Phương pháp liên kết'):
    axes[2].plot(sub['Số cụm k'], sub['Calinski-Harabasz'], marker=markers[ten_m], label=ten_m, color=palette[ten_m])
axes[2].set_title('Chỉ số Calinski-Harabasz (Càng cao càng tốt)', fontweight='bold')
axes[2].set_xlabel('Số lượng cụm (k)')
axes[2].set_ylabel('Chỉ số Calinski-Harabasz')
axes[2].legend()

plt.suptitle('So Sánh Các Chỉ Số Đo Lường Hiệu Quả Phân Cụm Theo Số Cụm k', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### 💡 Phân Tích Chuyên Sâu Kết Quả Grid Search

1. **Tại sao `Ward Linkage` vượt trội hoàn toàn so với `Complete` và `Average`?**
   - **Chỉ số Calinski-Harabasz của Ward cực cao** (đạt $1.656,75$ tại $k=4$), cao gấp **3 lần Complete** ($584,62$) và gấp **13 lần Average** ($125,23$). Điều này chứng minh các cụm do Ward tạo ra vô cùng chặt chẽ và phân tách rõ rệt trong không gian Euclidean.
   - **Sự cân bằng kích thước cụm (Cluster Balance)**: Ward phân bổ các cụm rất hợp lý (ở $k=4$: $215$, $319$, $980$, $1.202$ khách hàng). Ngược lại, `Average` và `Complete` bị hiện tượng cô lập ngoại lai (Outlier Splitting), tạo ra những cụm chỉ có 1 đến 4 khách hàng, hoàn toàn mất đi ý nghĩa kinh doanh.

2. **Tại sao chọn $k = 4$?**
   - Tại $k=4$, Silhouette của Ward duy trì mức tốt ($0,3089$), Davies-Bouldin đạt $1,044$, và Calinski-Harabasz đạt $1.656,75$.
   - Phù hợp hoàn hảo với **4 nhóm hành vi kinh điển trong bán lẻ**: *Khách hàng VIP*, *Khách hàng trung thành*, *Khách hàng tiềm năng*, và *Khách hàng có nguy cơ rời bỏ*.

In [ ]:
# Huấn luyện mô hình tối ưu Ward với k=4
optimal_k = 4
optimal_linkage = 'ward'

model = AgglomerativeClustering(n_clusters=optimal_k, linkage=optimal_linkage, metric='euclidean')
labels_dev = model.fit_predict(X_dev)

# Kết nối nhãn phân cụm với dữ liệu RFM gốc
rfm_segmented = rfm_dev.copy()
rfm_segmented['cluster'] = labels_dev
print(f'✓ Đã hoàn thành phân cụm cho {len(rfm_segmented):,} khách hàng với k={optimal_k}!')

## 4. Chân Dung Khách Hàng (Customer Persona Profiling)

Chúng ta tính toán các chỉ số thống kê trung vị (Median) và trung bình (Mean) của các giá trị RFM thực tế (chưa scale) cho từng nhóm để định danh chân dung kinh doanh.

In [ ]:
# Tính toán thống kê mô tả RFM theo cụm
summary = rfm_segmented.groupby('cluster').agg(
    customers=('CustomerID', 'count'),
    recency_median=('Recency', 'median'),
    frequency_median=('Frequency', 'median'),
    monetary_median=('Monetary', 'median'),
    monetary_total=('Monetary', 'sum'),
).reset_index()

summary['customer_share'] = summary['customers'] / len(rfm_segmented)
summary['revenue_share'] = summary['monetary_total'] / rfm_segmented['Monetary'].sum()

# Đặt tên tiếng Việt rõ ràng cho từng nhóm chân dung khách hàng
persona_names = {
    0: 'Khách hàng trung thành (Gắn kết đều, Giá trị khá)',
    1: 'Khách hàng có nguy cơ rời bỏ / Ngủ đông (Lâu không mua, Tần suất thấp)',
    2: 'Khách hàng VIP / Quán quân (Chi tiêu khủng, Mua thường xuyên)',
    3: 'Khách hàng tiềm năng (Mới mua gần đây, Đầy triển vọng)',
}
summary['persona'] = summary['cluster'].map(persona_names)
rfm_segmented['persona'] = rfm_segmented['cluster'].map(persona_names)

# Bảng hiển thị tiếng Việt định dạng đẹp mắt
summary_vi = pd.DataFrame({
    'Mã cụm': summary['cluster'],
    'Chân dung khách hàng': summary['persona'],
    'Số lượng khách': summary['customers'],
    'Tỷ lệ khách hàng': (summary['customer_share'] * 100).round(2).astype(str) + '%',
    'Recency trung vị (ngày)': summary['recency_median'].round(1),
    'Frequency trung vị (đơn)': summary['frequency_median'].round(1),
    'Monetary trung vị (£)': summary['monetary_median'].map('{:,.2f} £'.format),
    'Tổng doanh thu (£)': summary['monetary_total'].map('{:,.2f} £'.format),
    'Tỷ trọng doanh thu': (summary['revenue_share'] * 100).round(2).astype(str) + '%',
})

print('--- BẢNG HỒ SƠ PHÂN KHÚC KHÁCH HÀNG (RFM GỐC) ---')
display(summary_vi)

In [ ]:
# Trực quan hóa phân khúc bằng PCA 2D và Boxplots
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_dev)
rfm_segmented['PCA1'] = X_pca[:, 0]
rfm_segmented['PCA2'] = X_pca[:, 1]

# 1. Bản đồ phân cụm trên không gian PCA 2 chiều
plt.figure(figsize=(11, 6.5))
sns.scatterplot(
    data=rfm_segmented,
    x='PCA1',
    y='PCA2',
    hue='persona',
    palette='tab10',
    alpha=0.75,
    s=45,
)
plt.title(f'Bản Đồ Phân Cụm Khách Hàng (PCA 2D - Giải thích {pca.explained_variance_ratio_.sum()*100:.1f}% Phương Sai)', fontsize=13, fontweight='bold')
plt.xlabel(f'Thành phần chính 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'Thành phần chính 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

# 2. Phân bố RFM qua Boxplots (thang đo logarit cho Frequency và Monetary)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=rfm_segmented, x='cluster', y='Recency', ax=axes[0], hue='cluster', palette='Set2', legend=False)
axes[0].set_title('Recency (Số ngày kể từ lần mua gần nhất - Càng thấp càng tốt)')
axes[0].set_xlabel('Mã cụm (Cluster ID)')
axes[0].set_ylabel('Số ngày')

sns.boxplot(data=rfm_segmented, x='cluster', y='Frequency', ax=axes[1], hue='cluster', palette='Set2', legend=False)
axes[1].set_yscale('log')
axes[1].set_title('Frequency (Số lượng hóa đơn - Thang logarit)')
axes[1].set_xlabel('Mã cụm (Cluster ID)')
axes[1].set_ylabel('Số đơn hàng')

sns.boxplot(data=rfm_segmented, x='cluster', y='Monetary', ax=axes[2], hue='cluster', palette='Set2', legend=False)
axes[2].set_yscale('log')
axes[2].set_title('Monetary (Tổng giá trị chi tiêu £ - Thang logarit)')
axes[2].set_xlabel('Mã cụm (Cluster ID)')
axes[2].set_ylabel('Số tiền chi tiêu (£)')

plt.suptitle('Phân Phối Đặc Trưng RFM Theo Từng Cụm Khách Hàng', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 5. Đánh Giá Khả Năng Dự Báo Tương Lai (Out-of-Time Temporal Evaluation)

Một mô hình phân cụm tốt không chỉ có chỉ số nội suy đẹp, mà phải **phản ánh đúng hành vi mua sắm trong tương lai**.
Chúng ta kiểm tra kết quả hành vi trong **90 ngày tiếp theo** của từng cụm dựa trên dữ liệu thực tế:

In [ ]:
# Tải kết quả đánh giá ngoài thời gian từ thư mục reports
future_path = ROOT / 'reports' / 'temporal' / 'development' / 'future_cluster_profiles.csv'
if future_path.exists():
    future_df = pd.read_csv(future_path)
    future_df['persona'] = future_df['cluster'].map(persona_names)
    
    future_vi = pd.DataFrame({
        'Mã cụm': future_df['cluster'],
        'Chân dung khách hàng': future_df['persona'],
        'Số khách hàng': future_df['customers'],
        'Số người mua lại': future_df['repurchased'],
        'Tỷ lệ mua lại (%)': (future_df['repurchase_rate'] * 100).round(2).astype(str) + '%',
        'Doanh thu 90 ngày tới (£)': future_df['future_revenue'].map('{:,.2f} £'.format),
        'Doanh thu TB/Khách (£)': future_df['future_revenue_per_customer'].map('{:,.2f} £'.format),
        'Tỷ trọng doanh thu tương lai (%)': (future_df['future_revenue_share'] * 100).round(2).astype(str) + '%',
    })
    print('--- BẢNG THEO DÕI HÀNH VI MUA LẠI TRONG 90 NGÀY TƯƠNG LAI ---')
    display(future_vi)
    
    # Vẽ biểu đồ tỷ lệ mua lại và doanh thu tương lai
    fig, ax1 = plt.subplots(figsize=(10, 5))
    color = '#1f77b4'
    ax1.set_xlabel('Mã Cụm Khách Hàng', fontweight='bold')
    ax1.set_ylabel('Tỷ Lệ Mua Lại 90 Ngày Tới (Repurchase Rate)', color=color, fontweight='bold')
    bars = ax1.bar(future_df['cluster'].astype(str), future_df['repurchase_rate'], color=color, alpha=0.6, width=0.4, align='center', label='Tỷ lệ mua lại')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_ylim(0, 1.1)
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{height*100:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax2 = ax1.twinx()
    color = '#d62728'
    ax2.set_ylabel('Đóng Góp Doanh Thu Tương Lai (£)', color=color, fontweight='bold')
    line = ax2.plot(future_df['cluster'].astype(str), future_df['future_revenue'], color=color, marker='o', linewidth=2.5, label='Doanh thu tương lai')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('Hành Vi Mua Sắm 90 Ngày Tương Lai Theo Từng Phân Khúc Agglomerative', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Chưa tìm thấy future_cluster_profiles.csv. Hãy chạy script agglomerative.py trước!')

## 6. Đề Xuất Chiến Lược Marketing & Chăm Sóc Khách Hàng (Actionable Playbook)

| Cụm | Nhóm Khách Hàng | Đặc Điểm RFM | Hành Vi 90 Ngày Tới | Chiến Lược Marketing Khuyên Dùng |
| :---: | :--- | :--- | :--- | :--- |
| **Cụm 2** | **Khách hàng VIP / Quán quân** | $R=9$ ngày, $F=9$ đơn, $M=3.450£$ | **94.9% mua lại**, tạo ra **54.7% doanh thu** | **Chăm sóc đặc quyền VIP**: Quản lý tài khoản riêng (Dedicated account manager), quyền trải nghiệm sản phẩm mới trước, quà tri ân ngày lễ, tuyệt đối không spam giảm giá đại trà. |
| **Cụm 0** | **Khách hàng trung thành** | $R=27$ ngày, $F=3$ đơn, $M=924£$ | **64.6% mua lại**, tạo ra **28.6% doanh thu** | **Chương trình khách hàng thân thiết (Loyalty Program)**: Tích điểm đổi quà, gợi ý combo sản phẩm (Cross-sell/Up-sell) dựa trên lịch sử mua sắm để đẩy họ thành VIP. |
| **Cụm 3** | **Khách hàng tiềm năng** | $R=20$ ngày, $F=1$ đơn, $M=304£$ | **41.1% mua lại**, tạo ra **3.4% doanh thu** | **Kích hoạt đơn hàng thứ 2 (Onboarding & Nurturing)**: Gửi voucher chào mừng cho lần mua tiếp theo trong vòng 30 ngày, khảo sát mức độ hài lòng về sản phẩm đầu tiên. |
| **Cụm 1** | **Khách hàng có nguy cơ rời bỏ** | $R=97$ ngày, $F=1$ đơn, $M=261£$ | Chỉ **30.7% mua lại**, doanh thu chỉ **168£/người** | **Chiến dịch tái kích hoạt (Win-back Campaign)**: Gửi email 'Chúng tôi nhớ bạn' kèm ưu đãi giảm giá sâu có giới hạn thời gian (flash sale) để kéo họ quay lại; nếu không phản hồi thì loại khỏi danh sách email marketing thường xuyên để tiết kiệm chi phí gửi mail. |